In [1]:
!pip install google-cloud-vision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 7.3 MB/s eta 0:00:00


In [1]:
!pip install -q --upgrade google-cloud-vision transformers huggingface_hub accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 13.3 MB/s eta 0:00:00


In [17]:
from google.colab import files

uploaded = files.upload()

Saving allergen-scanner-506512-6b749d009330.json to allergen-scanner-506512-6b749d009330 (1).json


In [23]:
import os

# Correctly identify the JSON credentials file
json_file = 'allergen-scanner-506512-6b749d009330.json'

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = f"/content/{json_file}"

print("Credentials set to:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

Credentials set to: /content/allergen-scanner-506512-6b749d009330.json


In [19]:
import json

with open(os.environ["GOOGLE_APPLICATION_CREDENTIALS"], "r") as f:
    credentials = json.load(f)

print("Project:", credentials["project_id"])
print("Service account:", credentials["client_email"])

Project: allergen-scanner-506512
Service account: allergen-scanner-vision@allergen-scanner-506512.iam.gserviceaccount.com


## 🧀 Open Food Facts Ingredient Detection Model
Download and initialize the Open Food Facts sequence tagging model (`openfoodfacts/ingredient-detection`) fine-tuned from `xlm-roberta-large` to automatically isolate the ingredient list from packaging OCR text before allergen scanning.

In [20]:
# ============================================
# DOWNLOAD & LOAD OPEN FOOD FACTS INGREDIENT MODEL
# ============================================

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Use GPU if available in Colab
device = 0 if torch.cuda.is_available() else -1
model_id = "openfoodfacts/ingredient-detection"

print(f"Downloading and loading Open Food Facts model ({model_id})...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)

ingredient_extractor = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",
    stride=64,
    device=device
)

def extract_ingredients(raw_ocr_text, extractor):
    """
    Extracts the isolated ingredients section from raw packaging OCR text
    using the Open Food Facts sequence tagging model.
    """
    entities = extractor(raw_ocr_text)

    # Collect character spans for ingredient tokens (B-ING / I-ING)
    spans = []
    for ent in entities:
        group = ent.get("entity_group", ent.get("entity", ""))
        if "ING" in group:
            s, e = ent.get("start"), ent.get("end")
            if s is not None and e is not None:
                spans.append((s, e))

    if spans:
        # Merge overlapping or adjacent spans (within 5 characters)
        merged = []
        for s, e in sorted(spans, key=lambda x: x[0]):
            if not merged:
                merged.append([s, e])
            else:
                prev_s, prev_e = merged[-1]
                if s <= prev_e + 5:
                    merged[-1][1] = max(prev_e, e)
                else:
                    merged.append([s, e])

        extracted = [raw_ocr_text[s:e].strip() for s, e in merged]
        return "\n".join(extracted)

    print("⚠️ No distinct ingredient span detected; falling back to full OCR text.")
    return raw_ocr_text

print("✓ Open Food Facts model loaded and extraction pipeline ready!")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✓ Open Food Facts model loaded and extraction pipeline ready!


In [24]:
import pandas as pd
import re
from google.cloud import vision_v1 as vision

# ============================================
# 1. LOAD ALLERGEN DATASET
# ============================================

dataset_path = "/content/allergen_matching_dataset_reviewed_rtl.csv"

allergen_df = pd.read_csv(dataset_path)

# Clean the dataset
allergen_df["Allergen"] = (
    allergen_df["Allergen"]
    .astype(str)
    .str.strip()
)

allergen_df["Keyword"] = (
    allergen_df["Keyword"]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Allergen dataset loaded.")
print(f"Number of allergen keywords: {len(allergen_df)}")


# ============================================
# 2. UPLOAD PRODUCT IMAGE
# ============================================

from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

with open(filename, "rb") as f:
    content = f.read()

image = vision.Image(content=content)


# ============================================
# 3. SEND IMAGE TO GOOGLE VISION OCR
# ============================================

client = vision.ImageAnnotatorClient()

response = client.document_text_detection(image=image)

if response.error.message:
    raise Exception(response.error.message)

extracted_text = response.full_text_annotation.text

print("\n========== EXTRACTED TEXT ==========\n")
print(extracted_text)


# ============================================
# 4. EXTRACT INGREDIENT LIST (OPEN FOOD FACTS)
# ============================================

extracted_ingredients = extract_ingredients(extracted_text, ingredient_extractor)

print("\n========== EXTRACTED INGREDIENT LIST ==========\n")
print(extracted_ingredients)


# ============================================
# 5. NORMALIZE THE INGREDIENT TEXT
# ============================================

text = extracted_ingredients.lower()

# Replace line breaks and multiple spaces
text = re.sub(r"\s+", " ", text)

print("\n========== NORMALIZED TEXT ==========\n")
print(text)


# ============================================
# 6. CROSS-REFERENCE WITH ALLERGEN DATASET
# ============================================

detected_allergens = []

for _, row in allergen_df.iterrows():

    allergen = row["Allergen"]
    keyword = row["Keyword"]

    # Look for the keyword as a complete word
    pattern = r"\b" + re.escape(keyword) + r"\b"

    if re.search(pattern, text):
        detected_allergens.append({
            "allergen": allergen,
            "keyword": keyword
        })


# ============================================
# 7. REMOVE DUPLICATE ALLERGEN CATEGORIES
# ============================================

unique_allergens = {}

for item in detected_allergens:

    category = item["allergen"]

    if category not in unique_allergens:
        unique_allergens[category] = []

    unique_allergens[category].append(item["keyword"])


# ============================================
# 8. DISPLAY RESULTS
# ============================================

print("\n========== ALLERGEN RESULTS ==========\n")

if not unique_allergens:

    print("No allergens detected.")

else:

    print("⚠️ Potential allergens detected:\n")

    for allergen, keywords in unique_allergens.items():

        print(f"• {allergen}")

Allergen dataset loaded.
Number of allergen keywords: 337


Saving WhatsApp Image 2026-09-28 at 10.11.14 PM.jpeg to WhatsApp Image 2026-09-28 at 10.11.14 PM.jpeg

========== EXTRACTED TEXT ==========

EN Milk Chocolate Bar (43%) with Milky Cream Filling and Cocoa Biscuit,
Ingredients: Sugar, Full Fat Milk Powder (Cow), Non-Hydrogenated Vegetable fats
(Palm, Palm Kemel), Wheat Flour Cocoa Butter Cocoa Mass, Skimmed Milk Powder
(Cow), Whey Milk Powder (Cow), Lactose, Cocoa Powder, Emulsifier (E322), Com
Starch, Invert Sugar Syrup, Artificial Flavours (Ethyl Vanillin, Coconut, Butter, Yoghurt),
Natural flavour (Vanilla), Salt, Raising agents (E500, E4500), Acidity Regulator (E330).
Cocoa Solids: 30% minimum. Allergen statement: Contains Wheat (Gluten), Milk,
odour. Hazelnut and Soy. May also contain Egg, Nuts and Sesame. All our dairy products
are made with pasteurized cow's milk. It does not contain pork-based ingredients.
يحد
جوديفا / اصبع شوكولاتة الحليب (٤٣) محشو بكريمة الحليب وبسكويت الكا
القمح، زبدة الكاكاو كتلة الكاكاو، مسحوق حليب خالي الدس

/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:444: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(
